# 🧠 Step 1: ペルソナ定義 → LLM報酬パラメータ生成

```
入力: ペルソナ説明テキスト
  ↓ Claude API
出力: 報酬パラメータJSON (persona_rewards.json)
  ↓
Step2の学習ノートブックで使用
```

## 報酬パラメータの構造
```json
{
  "explore_bonus":    新規セル訪問ボーナス,
  "building_bonus":   建物セル滞在ボーナス,
  "road_bonus":       道路歩行ボーナス,
  "forward_bias":     前進行動への傾き,
  "revisit_penalty":  既訪問セルへのペナルティ,
  "violation_penalty":禁止エリア侵入ペナルティ,
  "goal_reward":      目的地到達報酬,
  "step_penalty":     毎ステップ生存コスト,
  "social_bonus":     他エージェント近接ボーナス,
  "description":      LLMによるペルソナ解釈
}
```

In [ ]:
!pip install anthropic -q
print('✓ Install complete')

In [ ]:
# セル2: インポート & Claude API設定
import anthropic, json, os
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
matplotlib.rcParams.update({
    'figure.facecolor':'#0a0c10','axes.facecolor':'#12161e',
    'text.color':'#c8d8e8','axes.labelcolor':'#c8d8e8',
    'xtick.color':'#4a5a6a','ytick.color':'#4a5a6a',
})

# Google Drive マウント (生成ファイルの保存先)
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/mesa_persona'
os.makedirs(SAVE_DIR, exist_ok=True)

# APIキー設定
# Colabのシークレット機能を使う場合:
# from google.colab import userdata
# ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
ANTHROPIC_API_KEY = 'sk-ant-...'  # ← ここにAPIキーを入力

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print('✓ Claude API client ready')

In [ ]:
# セル3: ペルソナ定義
# ── ここを自由に編集してください ──

PERSONAS = [
    {
        "id": "A",
        "name": "探索者タロウ",
        "description": "20歳、男性。新しい場所に積極的に訪れる探索好き。"
                        "マップの隅々まで歩き回り、同じ道は通りたがらない。"
                        "建物よりも道路を歩くこと自体が好き。"
    },
    {
        "id": "B",
        "name": "インドア花子",
        "description": "35歳、女性。外出は最小限にとどめたい慎重派。"
                        "できるだけ短い経路で目的の建物に到達したい。"
                        "知らない道を通るのは不安なので、"
                        "一度通った道を繰り返し使う傾向がある。"
    },
    {
        "id": "C",
        "name": "社交家ケンジ",
        "description": "28歳、男性。人と会うことが好きな社交的な性格。"
                        "他のエージェントの近くにいることを好み、"
                        "人の集まる場所（建物周辺）に積極的に向かう。"
                        "一人でいる時間が長いと不安になる。"
    },
    {
        "id": "D",
        "name": "ビジネスマン誠",
        "description": "45歳、男性。効率重視のビジネスパーソン。"
                        "常に最短経路で建物から建物へ移動する。"
                        "寄り道や探索は一切せず、目的地への直進を優先する。"
                        "時間は金なり、無駄なステップを嫌う。"
    },
    {
        "id": "E",
        "name": "観光客サラ",
        "description": "22歳、女性。初めて訪れた街を観光している外国人。"
                        "建物に立ち寄ることが好きで、各建物をじっくり見て回る。"
                        "道に迷いやすく、同じ場所をぐるぐると歩くことも多い。"
                        "でも新しい建物を発見したときの喜びは大きい。"
    },
]

print(f'定義済みペルソナ数: {len(PERSONAS)}')
for p in PERSONAS:
    print(f'  [{p["id"]}] {p["name"]}: {p["description"][:40]}...')

In [ ]:
# セル4: LLMによる報酬パラメータ生成関数

SYSTEM_PROMPT = """
あなたは強化学習の報酬設計の専門家です。
与えられたペルソナの説明から、都市シミュレーションにおける報酬パラメータを設計してください。

シミュレーションの設定:
- エージェントは30×30のグリッドマップを一人称視点で移動する
- セルタイプ: ROAD(道路), BUILDING(建物), TREE(木, 立入禁止), OTHER(その他, 立入禁止)
- 行動: 前進 / 左回転 / 右回転
- 目的: 建物から建物へ移動する
- 観測: 前方5方向のレイキャスト + 目的地方向

以下のJSON形式のみで回答してください。説明文は不要です:
{
  "explore_bonus": <float, 0.0~5.0, 未訪問セル到達時のボーナス>,
  "building_bonus": <float, 0.0~2.0, 建物セルにいるときの毎ステップボーナス>,
  "road_bonus": <float, 0.0~1.0, 道路セルにいるときの毎ステップボーナス>,
  "forward_bias": <float, 0.0~1.0, 前進行動への追加報酬>,
  "revisit_penalty": <float, 0.0~2.0, 既訪問セルへのペナルティ>,
  "violation_penalty": <float, 1.0~5.0, 禁止エリア侵入ペナルティ>,
  "goal_reward": <float, 5.0~30.0, 目的建物到達時の報酬>,
  "step_penalty": <float, 0.0~0.5, 毎ステップのコスト>,
  "social_bonus": <float, 0.0~3.0, 他エージェント近接ボーナス>,
  "stall_penalty": <float, 0.0~2.0, 同じ場所に留まり続けるペナルティ>,
  "description": "<string, ペルソナの行動特性をを1〜2文で説明>"
}
"""

def generate_reward_params(persona: dict) -> dict:
    """ペルソナ説明からLLMで報酬パラメータを生成"""
    user_msg = f"""
ペルソナID: {persona['id']}
名前: {persona['name']}
説明: {persona['description']}

このペルソナの行動特性に合った報酬パラメータをJSONで出力してください。
"""
    response = client.messages.create(
        model='claude-opus-4-5',
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user_msg}]
    )
    raw = response.content[0].text.strip()
    # JSONブロック抽出
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    params = json.loads(raw.strip())
    params['persona_id']   = persona['id']
    params['persona_name'] = persona['name']
    return params

print('generate_reward_params ✓')

In [ ]:
# セル5: 全ペルソナの報酬パラメータを生成
import time

all_rewards = {}

for persona in PERSONAS:
    print(f'[{persona["id"]}] {persona["name"]} を処理中...')
    try:
        params = generate_reward_params(persona)
        all_rewards[persona['id']] = params
        print(f'  ✓ goal_reward={params["goal_reward"]:.1f}  '
              f'explore={params["explore_bonus"]:.1f}  '
              f'social={params["social_bonus"]:.1f}')
        print(f'  説明: {params["description"]}')
    except Exception as e:
        print(f'  ❌ エラー: {e}')
        # フォールバック: デフォルト値
        all_rewards[persona['id']] = {
            'explore_bonus': 1.0, 'building_bonus': 0.2,
            'road_bonus': 0.3, 'forward_bias': 0.1,
            'revisit_penalty': 0.0, 'violation_penalty': 3.0,
            'goal_reward': 20.0, 'step_penalty': 0.1,
            'social_bonus': 0.0, 'stall_penalty': 0.8,
            'persona_id': persona['id'],
            'persona_name': persona['name'],
            'description': 'デフォルト設定',
        }
    time.sleep(1)  # APIレート制限対策

# 保存
save_path = f'{SAVE_DIR}/persona_rewards.json'
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(all_rewards, f, indent=2, ensure_ascii=False)
print(f'\n✓ 保存完了: {save_path}')

In [ ]:
# セル6: 報酬パラメータの可視化 (レーダーチャート)
import matplotlib.pyplot as plt
import numpy as np

RADAR_KEYS = [
    'explore_bonus', 'building_bonus', 'road_bonus',
    'forward_bias', 'social_bonus', 'goal_reward',
]
RADAR_LABELS = ['探索', '建物好き', '道路好き', '前進傾向', '社交性', '目標指向']
RADAR_MAX    = [5.0, 2.0, 1.0, 1.0, 3.0, 30.0]

COLORS = ['#ff3355','#00ccff','#33ff88','#ffee00','#ff7700']

N = len(RADAR_KEYS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0a0c10')
ax.set_facecolor('#0a0c10')

for i, (pid, params) in enumerate(all_rewards.items()):
    vals = [params.get(k,0)/m for k,m in zip(RADAR_KEYS, RADAR_MAX)]
    vals += vals[:1]
    color = COLORS[i % len(COLORS)]
    ax.plot(angles, vals, color=color, linewidth=2,
            label=f"{params['persona_name']}")
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_LABELS, color='#c8d8e8', fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%','50%','75%','100%'], color='#4a5a6a', fontsize=8)
ax.grid(color='#1e2530', linewidth=0.8)
ax.spines['polar'].set_color('#1e2530')
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
          facecolor='#12161e', edgecolor='#1e2530', labelcolor='#c8d8e8')
ax.set_title('ペルソナ別 報酬パラメータ比較', color='#00ddb4',
             fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/persona_radar.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0c10')
plt.show()
print('✓ レーダーチャート保存')

In [ ]:
# セル7: 生成された報酬パラメータの確認
print('='*60)
print('生成された報酬パラメータ一覧')
print('='*60)
for pid, p in all_rewards.items():
    print(f'\n【ペルソナ {pid}】{p["persona_name"]}')
    print(f'  解釈: {p["description"]}')
    print(f'  探索ボーナス    : {p["explore_bonus"]:.2f}')
    print(f'  建物ボーナス    : {p["building_bonus"]:.2f}')
    print(f'  道路ボーナス    : {p["road_bonus"]:.2f}')
    print(f'  前進バイアス    : {p["forward_bias"]:.2f}')
    print(f'  再訪ペナルティ  : {p["revisit_penalty"]:.2f}')
    print(f'  違反ペナルティ  : {p["violation_penalty"]:.2f}')
    print(f'  ゴール報酬      : {p["goal_reward"]:.2f}')
    print(f'  ステップコスト  : {p["step_penalty"]:.2f}')
    print(f'  社交ボーナス    : {p["social_bonus"]:.2f}')
    print(f'  滞留ペナルティ  : {p["stall_penalty"]:.2f}')

print(f'\n✓ persona_rewards.json を Step2 の学習ノートブックで使用します')